# Notebook 3: Full Transformer + Interpretability Tools

**Reading schedule — do BEFORE the indicated exercise:**

| Exercise | Read first |
|----------|-----------|
| Ex 1–3 (components) | nothing new |
| Ex 4 (SVD of W_OV) | [Elhage 2021](https://transformer-circuits.pub/2021/framework/index.html) §4 "Virtual Weights" |
| Ex 5 (induction heads) | [Olsson et al. 2022](https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html) §1–3 — what makes an induction head, K-composition, detection from attention patterns |
| Ex 6 (SAE) | [Bricken et al. 2023](https://transformer-circuits.pub/2023/monosemanticity/index.html) intro + §2 — why SAEs, the loss function, what encoder/decoder represent |


In [ ]:
import numpy as np

def softmax(z, axis=-1):
    z = z - np.max(z, axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

def check(name, got, expected=None, cond=None, rtol=1e-4, atol=1e-6):
    try:
        if cond is not None:
            passed = bool(cond(got))
        elif expected is not None:
            passed = np.allclose(np.array(got, dtype=float),
                                 np.array(expected, dtype=float), rtol=rtol, atol=atol)
        else:
            passed = True
        status = "PASS" if passed else "FAIL"
        print(f"  [{status}] {name}")
        if not passed and expected is not None:
            g, e = np.array(got), np.array(expected)
            print(f"         got shape {g.shape}, expected shape {e.shape}")
            if g.shape == e.shape and g.size <= 16:
                print(f"         got:      {g.ravel()}")
                print(f"         expected: {e.ravel()}")
        return passed
    except Exception as exc:
        print(f"  [ERROR] {name}: {type(exc).__name__}: {exc}")
        return False

print("Helper loaded. softmax and check() are available.")


## Ex 1: Layer normalization

In [ ]:
def layer_norm(x, eps=1e-5):
    """
    Normalize along last axis: mean 0, variance 1.
    x: (..., d). No learnable gamma/beta.
    """
    raise NotImplementedError("your code here")


In [ ]:
print("Ex 1: layer_norm")
x = np.random.randn(3, 8)
ln = layer_norm(x)
check("shape preserved", cond=lambda o: o.shape == x.shape, got=ln)
check("mean ~0", ln.mean(axis=-1), np.zeros(3), atol=1e-6)
check("var ~1", ln.var(axis=-1), np.ones(3), atol=1e-4)
x3 = np.random.randn(2, 5, 8)
ln3 = layer_norm(x3)
check("3D mean ~0", ln3.mean(axis=-1), np.zeros((2,5)), atol=1e-6)
# GOTCHA: constant vector — eps prevents division by zero
x_const = np.ones((4, 8))
check("constant → no NaN", cond=lambda o: np.all(np.isfinite(o)), got=layer_norm(x_const))


## Ex 2: GELU + MLP

In [ ]:
def gelu(x):
    """GELU (tanh approximation): 0.5x(1 + tanh(√(2/π)(x + 0.044715x³)))"""
    raise NotImplementedError("your code here")

def mlp(x, W1, W2, b1=None, b2=None):
    """
    x: (T, d_model). W1: (d_model, d_ff). W2: (d_ff, d_model).
    Returns (T, d_model).
    """
    raise NotImplementedError("your code here")


In [ ]:
print("Ex 2: gelu")
x = np.array([-2., -1., 0., 1., 2.])
g = gelu(x)
check("gelu(0) = 0", g[2], 0.0, atol=1e-6)
check("gelu(2) ≈ 2", g[4], 2.0, rtol=0.01)
check("gelu(-2) is small negative", cond=lambda v: -0.2 < v < 0, got=g[0])
print()
print("Ex 2: mlp")
T, dm, dff = 4, 8, 16
x = np.random.randn(T, dm)
W1 = np.random.randn(dm, dff); W2 = np.random.randn(dff, dm)
b1 = np.random.randn(dff);    b2 = np.random.randn(dm)
out = mlp(x, W1, W2)
check("no-bias shape (T,dm)", cond=lambda o: o.shape == (T,dm), got=out)
out_b = mlp(x, W1, W2, b1, b2)
check("bias changes output", cond=lambda o: not np.allclose(o, out), got=out_b)
check("bias shape still (T,dm)", cond=lambda o: o.shape == (T,dm), got=out_b)


## Ex 3: Full transformer layer (pre-norm)

Pre-norm convention (GPT-2 style):
```
x = x + attention_out(layer_norm(x))
x = x + mlp_out(layer_norm(x))
```


In [ ]:
def transformer_layer(x, W_Q, W_K, W_V, W_O_head, W1, W2, causal=True):
    """
    x: (T, d_model). Returns updated (T, d_model) residual stream.
    Single-head for simplicity.
    """
    raise NotImplementedError("your code here")


In [ ]:
print("Ex 3: transformer_layer")
T, dm, dk, dv, dff = 5, 8, 4, 4, 16
x = np.random.randn(T, dm)
W_Q = np.random.randn(dm,dk); W_K = np.random.randn(dm,dk)
W_V = np.random.randn(dm,dv); W_O_h = np.random.randn(dv,dm)
W1 = np.random.randn(dm,dff); W2 = np.random.randn(dff,dm)
out = transformer_layer(x, W_Q, W_K, W_V, W_O_h, W1, W2)
check("output shape (T,dm)", cond=lambda o: o.shape == (T,dm), got=out)
check("differs from input", cond=lambda o: not np.allclose(o, x), got=out)
check("output finite", cond=lambda o: np.all(np.isfinite(o)), got=out)


## Ex 4: SVD analysis of W_OV

**Read Elhage 2021 §4 first.**

The SVD of W_OV = U S Vhᵀ tells you what a head does:
- **Right singular vectors** (rows of Vh): input directions the head reads
- **Singular values S**: strength of each mode
- **Left singular vectors** (cols of U): output directions the head writes
- **Effective rank**: number of singular values needed to explain 90% of ‖W_OV‖²_F


In [ ]:
def analyze_wov(W_V, W_O_head, threshold=0.9):
    """
    Returns:
        W_OV: (d_model, d_model)
        singular_values: (d_model,) descending
        effective_rank: int — singular values needed for `threshold` of Frobenius norm²
    """
    raise NotImplementedError("your code here")


In [ ]:
print("Ex 4: analyze_wov")
dm, dv = 8, 4
W_V = np.random.randn(dm, dv); W_O_h = np.random.randn(dv, dm)
W_OV, sv, eff_rank = analyze_wov(W_V, W_O_h)
check("W_OV shape (dm,dm)", cond=lambda x: x.shape == (dm,dm), got=W_OV)
check("sv descending", cond=lambda s: np.all(s[:-1] >= s[1:]), got=sv)
check("sv non-negative", cond=lambda s: np.all(s >= 0), got=sv)
# Since W_V is (8,4), W_OV has rank at most 4
check("eff_rank ≤ min(dm,dv)=4", cond=lambda r: r <= 4, got=eff_rank)
# Rank-1 W_OV should have effective rank 1
W_V_r1 = np.outer(np.random.randn(dm), np.random.randn(dv))
W_O_r1 = np.outer(np.random.randn(dv), np.random.randn(dm))
_, _, er1 = analyze_wov(W_V_r1, W_O_r1)
check("rank-1 → eff_rank=1", er1, 1)


## Ex 5: Induction head detection

**Read Olsson et al. 2022 §1–3 first.**

On a repeated sequence [A₁…Aₙ A₁…Aₙ] (length 2n):
- At the second Aᵢ, an induction head attends to the first Bᵢ (the token that followed Aᵢ in the first copy)
- **Signature:** attention_weights[n+i, i+1] is high for i = 0…n−2
- Appears as a diagonal stripe **one above the main block diagonal**


In [ ]:
def induction_score(weights, n):
    """
    weights: (2n, 2n) attention weight matrix for one head.
    n: half-sequence length.
    Returns scalar in [0,1]: mean of weights[n+i, i+1] for i in 0..n-2.
    """
    raise NotImplementedError("your code here")


In [ ]:
print("Ex 5: induction_score")
n = 5; total = 2*n
# Perfect induction pattern
perfect = np.zeros((total, total))
for i in range(n-1):
    perfect[n+i, i+1] = 1.0
# fill non-induction rows to make valid probability distributions
perfect[2*n-1, :] = 1/total
for i in range(n): perfect[i, :i+1] = 1/(i+1)
sc = induction_score(perfect, n)
check("perfect induction scores > 0.9", cond=lambda s: s > 0.9, got=sc)
# Uniform attention should score low
uniform = np.ones((total,total)) / total
check("uniform scores low", cond=lambda s: s < 0.3, got=induction_score(uniform, n))
check("score in [0,1]", cond=lambda s: 0 <= s <= 1, got=sc)


## Ex 6: Sparse Autoencoder (SAE) forward pass

**Read Bricken et al. 2023 intro + §2 first.**

SAE architecture:
```
f(x) = ReLU(W_enc @ (x - b_dec) + b_enc)   # sparse features
x̂ = W_dec @ f(x) + b_dec                    # reconstruction
loss = MSE(x, x̂) + λ‖f(x)‖₁               # λ=0.01
```

The input is centered by b_dec before encoding — easy to forget this.


In [ ]:
def sae_forward(x, W_enc, b_enc, W_dec, b_dec, lam=0.01):
    """
    x: (T, d_model).
    W_enc: (d_model, d_hidden). W_dec: (d_hidden, d_model).
    Returns: features (T, d_hidden), x_hat (T, d_model), loss scalar.
    """
    raise NotImplementedError("your code here")


In [ ]:
print("Ex 6: sae_forward")
T, dm, dh = 3, 8, 32
x = np.random.randn(T, dm)
W_enc = np.random.randn(dm, dh)*0.1; b_enc = np.zeros(dh)
W_dec = np.random.randn(dh, dm)*0.1; b_dec = np.zeros(dm)
feats, xhat, loss = sae_forward(x, W_enc, b_enc, W_dec, b_dec)
check("features shape (T,dh)", cond=lambda f: f.shape == (T,dh), got=feats)
check("features ≥ 0 (ReLU)", cond=lambda f: np.all(f >= 0), got=feats)
check("x_hat shape (T,dm)", cond=lambda r: r.shape == (T,dm), got=xhat)
check("loss scalar positive", cond=lambda l: np.array(l).ndim == 0 and l > 0, got=loss)
# GOTCHA: centering — if x == b_dec, features should be ~zero
x_zero = np.tile(b_dec, (T,1))
f0, _, _ = sae_forward(x_zero, W_enc, b_enc, W_dec, b_dec)
check("x=b_dec → features ~0", cond=lambda f: np.all(f < 0.1), got=f0)
